# 🧠 Image Classifier — Exploration Notebook

This notebook walks through:
1. Loading and visualising the CIFAR-10 dataset
2. Building and training the CNN
3. Evaluating performance and visualising results
4. Generating Grad-CAM explanations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

import sys, os
sys.path.insert(0, os.path.abspath('..'))
from model.train import load_dataset, build_cnn

print('TF version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

## 1. Dataset Exploration

In [ ]:
(x_train, y_train), (x_test, y_test), cfg = load_dataset('cifar10')
class_names = cfg['class_names']

print(f'Train: {x_train.shape}')
print(f'Test:  {x_test.shape}')
print(f'Classes: {class_names}')

In [ ]:
# Visualise sample images
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('CIFAR-10 Samples', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i])
    ax.set_title(class_names[np.argmax(y_train[i])], fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 2. Build & Summarise the CNN

In [ ]:
model = build_cnn(
    input_shape=(32, 32, 3),
    num_classes=10,
    color_mode='rgb'
)
model.summary()

## 3. Quick Training Run (5 epochs for demo)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Grad-CAM Visualisation

In [ ]:
from model.evaluate import make_gradcam_heatmap, overlay_gradcam

# Find last conv layer
last_conv = next(
    layer.name for layer in reversed(model.layers)
    if isinstance(layer, keras.layers.Conv2D)
)
print('Using layer:', last_conv)

# Pick 5 test images
indices = np.random.choice(len(x_test), 5, replace=False)
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Grad-CAM: What the CNN Looks At', fontsize=13, fontweight='bold')

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)

for i, idx in enumerate(indices):
    img = x_test[idx]
    heatmap = make_gradcam_heatmap(np.expand_dims(img, 0), model, last_conv)
    overlay = overlay_gradcam(img, heatmap)
    
    color = 'green' if y_true[idx] == y_pred[idx] else 'red'
    axes[0, i].imshow(img)
    axes[0, i].set_title(
        f'True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}',
        fontsize=8, color=color
    )
    axes[0, i].axis('off')
    
    axes[1, i].imshow(overlay)
    axes[1, i].set_title('Grad-CAM', fontsize=8)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()